# MLOps Basics: From a Trained Model to a Working Product

**For:** anyone new to data science or machine learning. No MLOps experience needed.

In `train_model.ipynb` you trained two spam filters:

| Model | What it is like |
|---|---|
| **v1** | Looks accurate (86%), but lets almost all spam through |
| **v2** | Accurate *and* catches spam |

Training a model is only the start. **MLOps** (Machine Learning Operations) is everything that comes after: keeping track of your models, deciding which one users get, letting apps use it, updating it safely, and noticing when it stops working.

In this notebook you will do the **five big steps** of MLOps, each in a few lines of Python.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    S1["1 Save every version"] --> S2["2 Check quality<br/>before release"]
    S2 --> S3["3 Let apps<br/>use the model"]
    S3 --> S4["4 Retrain, and<br/>undo mistakes"]
    S4 --> S5["5 Watch for<br/>changes"]
    S5 -. "something changed" .-> S4
```

| Section | What to do |
|---|---|
| **Part A · Starter** | Read each step, then fill in the `# TODO` lines. They are short: usually one line. |
| **Part B · Solution** | The finished code and the answers. Use it to check your work, or if you get stuck. |

**Before you start:** run `python make_data.py` once, then run all of `train_model.ipynb`. Everything this notebook makes goes into a `mlops_basics/` folder.

> **Didn't finish `train_model.ipynb`?** No problem. Run `python make_solution_models.py` in a terminal (or `!python make_solution_models.py` in a notebook cell). It creates the two models this notebook needs, exactly as the solution would.

> Want more depth? `mlops_pipeline.ipynb` covers the same ideas in much more detail.

### Words you will meet

| Word | Plain meaning |
|---|---|
| **version** | One saved copy of a model, with a name like `v3` |
| **registry** | The list of all versions and how good each one is |
| **production** | The version that real users get right now |
| **quality gate** | Rules a model must pass before it reaches production |
| **promote** | Make a version the production version |
| **rollback** | Go back to an older, good version |
| **drift** | The real world slowly stops looking like the training data |
| **recall** | Of all the real spam, the share the model caught |

---
# Part A · Starter

Work from top to bottom: each step uses the step before it. If you see an error that says **"Finish the TODO"**, scroll up and fill in that cell.

> The setup cell empties `mlops_basics/` each time you run it. If you run it again, run the cells after it again too.

## Step 0 · Setup

Just run these two cells. The first loads the tools. The second creates the helper functions you will use in every step.

In [ ]:
import os
import shutil
import string

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

FEATURES = ["length", "punct"]
LABEL = "label"
FOLDER = "mlops_basics"
PRODUCTION = f"{FOLDER}/production.pkl"

for f in ["data/balanced.csv", "data/test_set.csv", "models/model_nb_v1.pkl", "models/model_nb_v2.pkl"]:
    assert os.path.exists(f), (f"{f} is missing. Run train_model.ipynb first, or run `python make_solution_models.py` "
                               "to get the solution models (run make_data.py first if data/ is empty).")

shutil.rmtree(FOLDER, ignore_errors=True)   # start fresh every time
os.makedirs(FOLDER)

test_set = pd.read_csv("data/test_set.csv")
print(f"Ready. Test set: {len(test_set)} messages.")

In [ ]:
def need(value, name):
    """Stop with a friendly message if a TODO was not filled in."""
    if value is None or value is ...:
        raise ValueError(f"Finish the TODO for `{name}` in this cell, then run it again.")


def score(model, data):
    """How good is a model? Returns accuracy and spam recall."""
    predictions = model.predict(data[FEATURES])
    return {
        "accuracy": round(accuracy_score(data[LABEL], predictions), 3),
        "recall": round(recall_score(data[LABEL], predictions, pos_label="spam", zero_division=0), 3),
    }


def extract_features(text):
    """Turn a message into the two numbers the model understands."""
    return {"length": len(text), "punct": sum(1 for ch in text if ch in string.punctuation)}


registry = []   # our list of versions: one dictionary per model


def save_version(model, trained_on):
    """Save a model as the next version and add it to the registry."""
    name = f"v{len(registry) + 1}"
    joblib.dump(model, f"{FOLDER}/{name}.pkl")
    registry.append({"version": name, "trained_on": trained_on, **score(model, test_set), "in_production": False})
    return name


def promote(name):
    """Make a version the production version."""
    shutil.copy(f"{FOLDER}/{name}.pkl", PRODUCTION)
    for row in registry:
        row["in_production"] = row["version"] == name
    print(f"  -> {name} is now in production")


def show_registry():
    return pd.DataFrame(registry)

## Step 1 · Save every version

**Why?** If you only keep "the latest model", then when it breaks you cannot go back, and you cannot even tell what changed.

**Think of it like save points in a video game.** Each version is a save point. The **registry** is the list of save points, with a note on how good each one is.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    M1["model v1"] --> R["registry<br/>v1, v2, ..."]
    M2["model v2"] --> R
    R --> Q["Which is best?<br/>Which is live?<br/>Where from?"]
```

Our registry is a simple Python list. Each version gets one row with:
- its **name** (`v1`, `v2`, ...)
- the **data** it was trained on
- its **scores** on the same test set, so versions are compared fairly
- whether it is **in production**

**Your turn:** add both models from `train_model.ipynb` to the registry.

In [ ]:
model_v1 = joblib.load("models/model_nb_v1.pkl")
model_v2 = joblib.load("models/model_nb_v2.pkl")

# TODO: save model_v1 as a version. It was trained on "data/sms_real_1k.csv"
# Hint: save_version(model_v1, trained_on="data/sms_real_1k.csv")
name1 = None
# TODO: save model_v2 as a version. It was trained on "data/balanced.csv"
name2 = None

need(name1, "name1")
need(name2, "name2")
show_registry()

**Q1.** Look at the registry. Which model would *you* pick for users, and which column told you?

## Step 2 · Check quality before release

**Why?** A broken model should never reach users. So before a model goes to production, it must pass a **quality gate**: a few simple rules, checked automatically.

**Think of it like a driving test.** You do not get a licence because you *feel* ready. You pass a test first.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    C["new model"] --> G{"quality<br/>gate"}
    G -- "pass" --> P["production"]
    G -- "fail" --> K["keep the<br/>old model"]
```

### 2a · A gate that only checks accuracy

Let's start with the simplest rule: *accuracy must be at least 85%.*

In [ ]:
def gate_accuracy_only(scores):
    # TODO: return True if scores["accuracy"] is at least 0.85
    return None


for row in registry:
    result = gate_accuracy_only(row)
    need(result, "gate_accuracy_only")
    print(f"{row['version']}: accuracy {row['accuracy']:.1%}, recall {row['recall']:.1%} -> {'PASS' if result else 'FAIL'}")

**Q2.** Did v1 pass? Is that a good thing?

### 2b · A better gate

Add a second rule: **recall must be at least 80%**. The model must pass *both* rules.

In [ ]:
def gate(scores):
    # TODO: return True only if accuracy >= 0.85 AND recall >= 0.80
    # Hint: use `and` to combine the two rules
    return None


for row in registry:
    result = gate(row)
    need(result, "gate")
    print(f"{row['version']}: accuracy {row['accuracy']:.1%}, recall {row['recall']:.1%} -> {'PASS' if result else 'FAIL'}")

### 2c · Release v2

v2 passed, so **promote** it. That copies its file to `production.pkl`, the one file the app uses.

In [ ]:
# TODO: promote "v2"
...
show_registry()

## Step 3 · Let apps use the model

**Why?** A model in a notebook helps no one. Apps (a website, a phone, another program) need a simple way to ask: *"Is this message spam?"*

We wrap the model in one function. In real life this function would sit behind a web address (an **API**), so any app can call it.

```mermaid
sequenceDiagram
    participant App
    participant F as predict()
    participant M as production.pkl
    App->>F: "WIN a free prize!!!"
    F->>M: length=19, punct=3
    M-->>F: spam
    F-->>App: spam
```

Notice that the app never needs to know *which* version is in production. You can swap the model without changing the app.

In [ ]:
def predict(message):
    if not os.path.exists(PRODUCTION):
        raise ValueError("No model is in production yet. Finish step 2c (promote v2), then run this cell again.")
    model = joblib.load(PRODUCTION)
    # TODO: turn the message into features, then ask the model
    # Hint: features = extract_features(message)
    # Hint: model.predict(pd.DataFrame([features]))[0]
    answer = None
    need(answer, "answer")
    return answer


for message in ["See you at lunch",
                "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
                "WINNER!! Claim your free prize now!!!"]:
    print(f"{predict(message):>5}  <-  {message[:55]}")

**To put it online for real**, the repo already has the pieces: `app.py` is a small web page around the model, and the `Dockerfile` packages everything into a container that runs the same on any computer (slide 9):

```bash
docker build -t spam-lab .
docker run -p 8501:8501 spam-lab
```

**Q3.** The last message is clearly spam, but the model says "ham". Is that an MLOps problem or a model problem?

## Step 4 · Retrain, and undo mistakes

**Why?** New data arrives every week. Retraining by hand is slow and easy to get wrong. A **pipeline** does it the same way every time:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    N["new data"] --> T["train a new model"]
    T --> S["save it as<br/>a new version"]
    S --> G{"quality gate"}
    G -- "pass" --> P["promote"]
    G -- "fail" --> K["keep the old one"]
```

The pipeline below is almost finished. Fill in the last line.

In [ ]:
def retrain(new_data_file, gate_rule):
    data = pd.concat([pd.read_csv("data/balanced.csv"), pd.read_csv(new_data_file)])
    model = LogisticRegression(max_iter=1000).fit(data[FEATURES], data[LABEL])
    name = save_version(model, trained_on=f"balanced + {new_data_file}")
    scores = registry[-1]
    print(f"{name}: accuracy {scores['accuracy']:.1%}, recall {scores['recall']:.1%}")

    passed = gate_rule(scores)
    need(passed, "gate_rule")
    if passed:
        # TODO: promote the new version
        ...
    else:
        print("  -> gate FAILED, production is unchanged")
    return name

### 4a · A normal week

In [ ]:
retrain("data/new_week_ok.csv", gate)
show_registry()

### 4b · A bad week, with the weak gate

This week's data has a bug: most spam was labeled "ham" by mistake. What happens if the pipeline still used the **accuracy-only** gate?

In [ ]:
retrain("data/new_week_bad.csv", gate_accuracy_only)
show_registry()

A broken model (recall 12%) just reached users. Time to **roll back**.

### 4c · Roll back

Because we saved every version, the good model from the normal week is still there. Rolling back just means promoting it again.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    BAD["v4 broken<br/>in production"] -- "rollback" --> GOOD["v3 good<br/>in production"]
```

In [ ]:
# TODO: roll back by promoting "v3", the version from the normal week
...
show_registry()

### 4d · The same bad week, with the better gate

In [ ]:
retrain("data/new_week_bad.csv", gate)
show_registry()

**Q4.** The bad model (v5) was blocked this time, but we still saved it. Why keep a model we will never use?

## Step 5 · Watch for changes (drift)

**Why?** A model learns from the past but is used in the future. If the world changes, the model quietly gets worse, and nobody tells you.

**Example:** spammers learn that long messages get caught, so they start sending **short** ones. The model was trained on long spam, so it misses them.

The good news: you can spot this **without knowing the right answers**. Just compare what new messages look like with what the training data looked like.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    OLD["training data<br/>average length"] --> C{"big change?"}
    NEW["this week<br/>average length"] --> C
    C -- "yes" --> A["ALERT:<br/>drift"]
    C -- "no" --> OK["all good"]
```

We call a change of more than **25%** drift.

In [ ]:
training_data = pd.read_csv("data/balanced.csv")


def check_drift(new_data_file):
    new = pd.read_csv(new_data_file)
    print(new_data_file)
    for feature in FEATURES:
        before = training_data[feature].mean()
        after = new[feature].mean()
        # TODO: compute the change as a fraction: (after - before) / before
        change = None
        need(change, "change")
        flag = "DRIFT!" if abs(change) > 0.25 else "ok"
        print(f"  average {feature:<6} {before:6.1f} -> {after:6.1f}  ({change:+.0%})  {flag}")


check_drift("data/new_week_ok.csv")
check_drift("data/new_week_drift.csv")

### 5b · Did the drift hurt the model?

A few days later someone labels the drifted messages, so we can check how production really did:

In [ ]:
production_model = joblib.load(PRODUCTION)
pd.DataFrame({
    "normal week": score(production_model, pd.read_csv("data/new_week_ok.csv")),
    "drift week": score(production_model, pd.read_csv("data/new_week_drift.csv")),
}).T

**Q5.** Which features drifted, and in which direction? What happened to recall in the drift week?

### What happens next? Close the loop

Drift sends you back to step 4: collect new labeled data, retrain, check the gate, release. That is why MLOps is drawn as a **loop**, not a line.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    W["watch"] -- "drift" --> R["retrain"]
    R --> G{"gate"}
    G -- "pass" --> REL["release"]
    REL --> W
    G -- "fail" --> H["ask a human"]
```

Sometimes retraining is not enough. Here, short spam and normal texts look almost the same in `length` and `punct`, so the real fix is a smarter model that reads the **words**. That is where deep learning comes in.

## Wrap-up

| Step | What you did | Real-world tools that do this |
|---|---|---|
| 1 Save every version | A registry of v1, v2, ... with scores | MLflow, SageMaker Model Registry |
| 2 Check quality | A gate with two rules | Checks inside CI/CD (GitHub Actions) |
| 3 Let apps use it | A `predict()` function | Docker, FastAPI, serverless functions |
| 4 Retrain and undo | A pipeline, and a rollback in one line | Airflow, GitHub Actions |
| 5 Watch for drift | Compare averages with the training data | Evidently, cloud monitoring dashboards |

**Q6.** In one sentence each: what would go wrong without step 1? Without step 2? Without step 5?

### Five things to remember

1. **Save every version**, with what data made it and how good it was.
2. **Never trust accuracy alone.** Check the metric that matters for the job (here, recall).
3. **Gate before you release.** Bad models should be stopped automatically.
4. **Keep old versions** so you can roll back in seconds.
5. **Keep watching.** The world changes, and your model will not tell you on its own.

---
# Part B · Solution

This part runs on its own from top to bottom. The explanations are in Part A.

## Step 0 · Setup

In [ ]:
import os
import shutil
import string

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

FEATURES = ["length", "punct"]
LABEL = "label"
FOLDER = "mlops_basics"
PRODUCTION = f"{FOLDER}/production.pkl"

for f in ["data/balanced.csv", "data/test_set.csv", "models/model_nb_v1.pkl", "models/model_nb_v2.pkl"]:
    assert os.path.exists(f), (f"{f} is missing. Run train_model.ipynb first, or run `python make_solution_models.py` "
                               "to get the solution models (run make_data.py first if data/ is empty).")

shutil.rmtree(FOLDER, ignore_errors=True)   # start fresh every time
os.makedirs(FOLDER)

test_set = pd.read_csv("data/test_set.csv")
print(f"Ready. Test set: {len(test_set)} messages.")

In [ ]:
def need(value, name):
    """Stop with a friendly message if a TODO was not filled in."""
    if value is None or value is ...:
        raise ValueError(f"Finish the TODO for `{name}` in this cell, then run it again.")


def score(model, data):
    """How good is a model? Returns accuracy and spam recall."""
    predictions = model.predict(data[FEATURES])
    return {
        "accuracy": round(accuracy_score(data[LABEL], predictions), 3),
        "recall": round(recall_score(data[LABEL], predictions, pos_label="spam", zero_division=0), 3),
    }


def extract_features(text):
    """Turn a message into the two numbers the model understands."""
    return {"length": len(text), "punct": sum(1 for ch in text if ch in string.punctuation)}


registry = []   # our list of versions: one dictionary per model


def save_version(model, trained_on):
    """Save a model as the next version and add it to the registry."""
    name = f"v{len(registry) + 1}"
    joblib.dump(model, f"{FOLDER}/{name}.pkl")
    registry.append({"version": name, "trained_on": trained_on, **score(model, test_set), "in_production": False})
    return name


def promote(name):
    """Make a version the production version."""
    shutil.copy(f"{FOLDER}/{name}.pkl", PRODUCTION)
    for row in registry:
        row["in_production"] = row["version"] == name
    print(f"  -> {name} is now in production")


def show_registry():
    return pd.DataFrame(registry)

## Step 1 · Save every version

In [ ]:
model_v1 = joblib.load("models/model_nb_v1.pkl")
model_v2 = joblib.load("models/model_nb_v2.pkl")

name1 = save_version(model_v1, trained_on="data/sms_real_1k.csv")
name2 = save_version(model_v2, trained_on="data/balanced.csv")

need(name1, "name1")
need(name2, "name2")
show_registry()

**A1.** **v2.** Both have high accuracy (86% and 99%), so the `accuracy` column alone makes them look similar. The `recall` column shows the real difference: v1 catches 4% of spam, v2 catches 100%.

## Step 2 · Check quality before release

In [ ]:
def gate_accuracy_only(scores):
    return scores["accuracy"] >= 0.85


for row in registry:
    result = gate_accuracy_only(row)
    need(result, "gate_accuracy_only")
    print(f"{row['version']}: accuracy {row['accuracy']:.1%}, recall {row['recall']:.1%} -> {'PASS' if result else 'FAIL'}")

**A2.** Yes, v1 passed (85.6% is above 85%), and that is **bad**. v1 catches only 4% of spam. Most messages are not spam, so a model that almost always says "not spam" still gets high accuracy. A gate that only checks accuracy would let this useless model reach users.

### 2b · A better gate

In [ ]:
def gate(scores):
    return scores["accuracy"] >= 0.85 and scores["recall"] >= 0.80


for row in registry:
    result = gate(row)
    need(result, "gate")
    print(f"{row['version']}: accuracy {row['accuracy']:.1%}, recall {row['recall']:.1%} -> {'PASS' if result else 'FAIL'}")

### 2c · Release v2

In [ ]:
promote("v2")
show_registry()

## Step 3 · Let apps use the model

In [ ]:
def predict(message):
    if not os.path.exists(PRODUCTION):
        raise ValueError("No model is in production yet. Finish step 2c (promote v2), then run this cell again.")
    model = joblib.load(PRODUCTION)
    features = extract_features(message)
    answer = model.predict(pd.DataFrame([features]))[0]
    need(answer, "answer")
    return answer


for message in ["See you at lunch",
                "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
                "WINNER!! Claim your free prize now!!!"]:
    print(f"{predict(message):>5}  <-  {message[:55]}")

**A3.** A **model** problem. MLOps delivered the model correctly. The model itself only looks at length and punctuation, and that short message looks like a normal text. Good MLOps cannot fix a model that cannot see the words, but monitoring (step 5) can help you *notice* problems like this.

## Step 4 · Retrain, and undo mistakes

In [ ]:
def retrain(new_data_file, gate_rule):
    data = pd.concat([pd.read_csv("data/balanced.csv"), pd.read_csv(new_data_file)])
    model = LogisticRegression(max_iter=1000).fit(data[FEATURES], data[LABEL])
    name = save_version(model, trained_on=f"balanced + {new_data_file}")
    scores = registry[-1]
    print(f"{name}: accuracy {scores['accuracy']:.1%}, recall {scores['recall']:.1%}")

    passed = gate_rule(scores)
    need(passed, "gate_rule")
    if passed:
        promote(name)
    else:
        print("  -> gate FAILED, production is unchanged")
    return name

### 4a · A normal week

In [ ]:
retrain("data/new_week_ok.csv", gate)
show_registry()

### 4b · A bad week, with the weak gate

In [ ]:
retrain("data/new_week_bad.csv", gate_accuracy_only)
show_registry()

### 4c · Roll back

In [ ]:
promote("v3")
show_registry()

### 4d · The same bad week, with the better gate

In [ ]:
retrain("data/new_week_bad.csv", gate)
show_registry()

**A4.** It is **evidence**. The registry shows v4 and v5 both came from `new_week_bad.csv`, which points straight at the data bug. Keeping every version also means you can always roll back, and you can compare versions over time. Deleting the bad ones would erase what happened.

## Step 5 · Watch for changes (drift)

In [ ]:
training_data = pd.read_csv("data/balanced.csv")


def check_drift(new_data_file):
    new = pd.read_csv(new_data_file)
    print(new_data_file)
    for feature in FEATURES:
        before = training_data[feature].mean()
        after = new[feature].mean()
        change = (after - before) / before
        need(change, "change")
        flag = "DRIFT!" if abs(change) > 0.25 else "ok"
        print(f"  average {feature:<6} {before:6.1f} -> {after:6.1f}  ({change:+.0%})  {flag}")


check_drift("data/new_week_ok.csv")
check_drift("data/new_week_drift.csv")

### 5b · Did the drift hurt the model?

In [ ]:
production_model = joblib.load(PRODUCTION)
pd.DataFrame({
    "normal week": score(production_model, pd.read_csv("data/new_week_ok.csv")),
    "drift week": score(production_model, pd.read_csv("data/new_week_drift.csv")),
}).T

**A5.** Both dropped a lot: average length fell by about half (about 100 → 47 characters) and average punctuation by about 40%. Recall fell from about 99% to under 2%: the model now misses almost all spam. The drift check warned us **before** anyone labeled a single message.

**A6.** **Without step 1**, you could not roll back the broken v4, because the good v3 would be gone. **Without step 2**, useless models like v1 and the bad-week models would reach users automatically. **Without step 5**, spam would pour in for weeks after the spammers changed tactics, and you would only find out from angry users.